In [1]:
'''
Depth-Aware Image Enhancement for Night-Time Outdoor Scenes
Uses scene depth to perform spatially different enhancement
with real-time
'''
import numpy as np, cv2, ipywidgets as widgets
from IPython.display import display


def estimate_depth(bgr, omega=0.95, k=15):
    """Night-scene relative depth: 0 = near, 1 = far.
    Also returns transmission t and atmospheric light A for glow removal."""
    img = bgr.astype(np.float32) / 255.0
    ker = cv2.getStructuringElement(cv2.MORPH_RECT, (k, k))

    # element-wise min over the 3 channels (np.minimum is safer than cv2.min)
    dc = np.minimum(np.minimum(img[..., 0], img[..., 1]), img[..., 2])
    dc = cv2.erode(dc, ker)

    flat = dc.ravel()
    n    = max(1, int(0.001 * flat.size))
    idx  = np.argpartition(flat, -n)[-n:]                 # brightest 0.1 %
    A    = np.clip(img.reshape(-1, 3)[idx].mean(0), 0.3, 1.0)  # shape (3,)

    norm = img / A
    t = 1.0 - omega * cv2.erode(
        np.minimum(np.minimum(norm[..., 0], norm[..., 1]), norm[..., 2]), ker)
    t = cv2.GaussianBlur(np.clip(t, 0.1, 1.0), (0, 0), 7)   # smooth -> no halos

    d = -np.log(t + 1e-6)
    d = (d - d.min()) / (float(np.ptp(d)) + 1e-6)
    return d.astype(np.float32), t.astype(np.float32), A.astype(np.float32)

def enhance(img, depth, t, A, near_gain=1.9, far_gain=1.15, dehaze=0.75,
            local_contrast=2.0, sharpen=1.0, gamma=0.85, denoise=0.5):
    near = (1.0 - depth)[..., None]                        # 1 = close

    # (a) transmission-based dehaze / lamp-glow suppression (far only)
    deh = (img - A) / np.clip(t, 0.25, 1.0)[..., None] + A
    w   = dehaze * depth[..., None]
    img = np.clip(img * (1 - w) + deh * w, 0, 1)

    # (b) depth-dependent exposure
    gain = far_gain + (near_gain - far_gain) * near
    img  = np.clip(img * gain, 0, 1)

    # (c) depth-dependent gamma (lift shadows up close, keep sky dark)
    img = np.power(np.clip(img, 1e-6, 1.0), 1.0 + (gamma - 1.0) * near)

    # (d) local contrast (CLAHE on L channel)
    lab = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=float(max(0.1, local_contrast)),
                            tileGridSize=(8, 8))
    lab[..., 0] = clahe.apply(lab[..., 0])
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR).astype(np.float32) / 255.0

    # (e) far = noisier -> bilateral denoise weighted by depth
    if denoise > 0:
        sm  = cv2.bilateralFilter((img * 255).astype(np.uint8), 7,
                                  40 * denoise, 7).astype(np.float32) / 255.0
        mix = (denoise * depth)[..., None]
        img = img * (1 - mix) + sm * mix

    # (f) unsharp mask weighted to nearby detail
    if sharpen > 0:
        blur = cv2.GaussianBlur(img, (0, 0), 2.0)
        img  = np.clip(img + (sharpen * near) * (img - blur), 0, 1)

    return img

def synthetic_night(h=360, w=640, seed=3):
    rng = np.random.default_rng(seed)
    img = np.zeros((h, w, 3), np.float32)
    hz  = int(0.45 * h)

    img[:hz] = np.linspace(0.16, 0.05, hz)[:, None, None] * np.array([1.0, 0.55, 0.35])
    img[hz:] = np.linspace(0.03, 0.14, h - hz)[:, None, None]

    for x0 in (0.18, 0.82):                                # sidewalk edges
        cv2.line(img, (int(x0 * w), h - 1), (w // 2, hz), (0.25, 0.25, 0.28), 2)
    for i in range(6):                                     # road markings
        y1 = int(hz + (h - hz) * (i / 6) ** 1.6)
        y2 = int(hz + (h - hz) * ((i + .5) / 6) ** 1.6)
        cv2.line(img, (w // 2, y1), (w // 2, y2), (0.3, 0.3, 0.32), 2)

    x = 0                                                  # skyline
    while x < w:
        bw, bh = rng.integers(40, 110), rng.integers(30, 90)
        cv2.rectangle(img, (x, hz - bh), (min(x + bw, w), hz), (0.035, 0.035, 0.045), -1)
        x += bw + rng.integers(5, 25)

    yy, xx = np.mgrid[0:h, 0:w]
    for lx in (0.25, 0.55, 0.85):                          # lamps + glow
        px, py = int(lx * w), int(hz - 0.18 * h)
        img += (np.exp(-((xx - px) ** 2 + (yy - py) ** 2) / (2 * (0.05 * w) ** 2))
                )[..., None] * np.array([0.25, 0.75, 1.0]) * 0.85
        cv2.circle(img, (px, py), 4, (0.6, 0.9, 1.0), -1)
        cv2.line(img, (px, py), (px, hz + 20), (0.08, 0.08, 0.09), 3)

    img += rng.normal(0, 0.012, img.shape).astype(np.float32)
    img += rng.normal(0, 0.030, img.shape).astype(np.float32) * \
           (1 - np.clip(img.mean(2, keepdims=True), 0, 1))  # sensor noise
    return np.clip(img, 0, 1).astype(np.float32)


def to_png(bgr):
    return cv2.imencode('.png', (np.clip(bgr, 0, 1) * 255).astype(np.uint8))[1].tobytes()

def depth_png(d):
    colored = cv2.applyColorMap((d * 255).astype(np.uint8), cv2.COLORMAP_INFERNO)
    return to_png(colored.astype(np.float32) / 255.0)

state = {'img': synthetic_night()}
cache = {}

S = lambda **k: widgets.FloatSlider(**k, continuous_update=True,
                                    layout=widgets.Layout(width='260px'))
sl = {
    'near_gain':      S(value=1.90, min=1.0, max=3.5, step=0.05, description='Near gain'),
    'far_gain':       S(value=1.15, min=0.6, max=2.5, step=0.05, description='Far gain'),
    'dehaze':         S(value=0.75, min=0.0, max=1.0, step=0.05, description='Glow removal'),
    'local_contrast': S(value=2.00, min=0.1, max=5.0, step=0.10, description='Local contrast'),
    'sharpen':        S(value=1.00, min=0.0, max=3.0, step=0.10, description='Sharpen (near)'),
    'gamma':          S(value=0.85, min=0.4, max=1.6, step=0.05, description='Shadow lift'),
    'denoise':        S(value=0.50, min=0.0, max=1.0, step=0.05, description='Denoise (far)'),
}
width_sl = widgets.IntSlider(value=480, min=240, max=960, step=80,
                             description='Process width',
                             layout=widgets.Layout(width='260px'))

img_in, img_d, img_out = (widgets.Image(format='png', width='380') for _ in range(3))
upload = widgets.FileUpload(accept='image/*', description='Upload night photo')

def refresh(*_):
    img = state['img']
    w   = width_sl.value
    h   = max(1, int(img.shape[0] * w / img.shape[1]))
    key = (id(img), w, h)
    if cache.get('key') != key:                    # depth only depends on image
        small = cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)
        d, t, A = estimate_depth(small)
        cache.update(key=key, small=small, d=d, t=t, A=A)
    c = cache
    out = enhance(c['small'], c['d'], c['t'], c['A'],
                  **{k: v.value for k, v in sl.items()})
    img_in.value  = to_png(c['small'])
    img_d.value   = depth_png(c['d'])
    img_out.value = to_png(out)

def on_upload(change):
    val = upload.value
    if not val:
        return
    item = next(iter(val.values())) if isinstance(val, dict) else val[0]
    bgr  = cv2.imdecode(np.frombuffer(item['content'], np.uint8), cv2.IMREAD_COLOR)
    if bgr is None:
        return
    state['img'] = bgr.astype(np.float32) / 255.0
    cache.clear()
    refresh()

upload.observe(on_upload, 'value')
for s in list(sl.values()) + [width_sl]:
    s.observe(refresh, 'value')

lbl = lambda t: widgets.HTML(
    f"<div style='text-align:center;font-weight:600;color:#e8e8e8;"
    f"background:#1b1b1f;padding:4px;border-radius:6px'>{t}</div>")

gui = widgets.VBox([
    widgets.HTML("<h3 style='color:#e8e8e8;margin:4px'>🌙 Depth-Aware Night Image Enhancement</h3>"),
    upload,
    widgets.HBox([widgets.VBox([lbl('Night input'),     img_in]),
                  widgets.VBox([lbl('Estimated depth'), img_d]),
                  widgets.VBox([lbl('Enhanced'),        img_out])]),
    widgets.HBox(list(sl.values()) + [width_sl]),
], layout=widgets.Layout(padding='12px', background='#101014'))

display(gui)
refresh()